# Clase 125 — TFRecord

**TFRecord** es el formato binario nativo de TF, optimizado para datasets
grandes que no caben en RAM. Serializamos `tf.train.Example`, escribimos con
`TFRecordWriter` y parseamos con `tf.io.parse_single_example`.

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. Helpers de `Feature` (BytesList / Int64List / FloatList)

In [ ]:
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(42)

def _bytes_feature(valor):
    if isinstance(valor, type(tf.constant(0))):
        valor = valor.numpy()
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[valor]))

def _int64_feature(valor):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[valor]))

def _float_feature(valor):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[valor]))

print("helpers de Feature listos: bytes / int64 / float")

## 2. Serializar un `tf.train.Example`

In [ ]:
(X_tr, y_tr), _ = keras.datasets.fashion_mnist.load_data()

def serializar_ejemplo(imagen, etiqueta):
    imagen_bytes = tf.io.serialize_tensor(tf.constant(imagen, dtype=tf.uint8))
    ejemplo = tf.train.Example(features=tf.train.Features(feature={
        "image": _bytes_feature(imagen_bytes),
        "label": _int64_feature(int(etiqueta)),
    }))
    return ejemplo.SerializeToString()

serial = serializar_ejemplo(X_tr[0], y_tr[0])
print("Example serializado a", len(serial), "bytes")

## 3. Escribir TFRecord en shards con `TFRecordWriter`

In [ ]:
N_SHARDS = 5
n = 2000                                   # subconjunto para el ejemplo
for s in range(N_SHARDS):
    ruta = f"fashion-{s:05d}-of-{N_SHARDS:05d}.tfrecord"
    with tf.io.TFRecordWriter(ruta) as w:
        for i in range(s, n, N_SHARDS):
            w.write(serializar_ejemplo(X_tr[i], y_tr[i]))
print("shards escritos:", sorted(glob.glob("fashion-*.tfrecord")))

## 4. Leer y parsear con `parse_single_example` + `FixedLenFeature`

In [ ]:
feature_spec = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parsear(serializado):
    ej = tf.io.parse_single_example(serializado, feature_spec)
    imagen = tf.io.parse_tensor(ej["image"], out_type=tf.uint8)
    imagen = tf.reshape(imagen, (28, 28))
    return imagen, ej["label"]

ds = tf.data.TFRecordDataset(glob.glob("fashion-*.tfrecord")).map(parsear)
img, lab = next(iter(ds))
print("parseado:", img.shape, "| label:", int(lab))

## 5. Reads paralelos con `interleave` + compresión GZIP

In [ ]:
archivos = tf.data.Dataset.list_files("fashion-*.tfrecord")
ds_paralelo = archivos.interleave(
    lambda f: tf.data.TFRecordDataset(f),
    cycle_length=N_SHARDS, num_parallel_calls=tf.data.AUTOTUNE)
print("registros leídos en paralelo:", sum(1 for _ in ds_paralelo))

# escribir comprimido con GZIP
opciones = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("fashion.gz.tfrecord", opciones) as w:
    w.write(serializar_ejemplo(X_tr[0], y_tr[0]))
ds_gz = tf.data.TFRecordDataset("fashion.gz.tfrecord", compression_type="GZIP")
print("registros GZIP:", sum(1 for _ in ds_gz))

## 6. `FixedLenFeature` vs `VarLenFeature`

In [ ]:
# VarLenFeature -> SparseTensor (longitud variable por record)
spec_var = {"tokens": tf.io.VarLenFeature(tf.int64)}
ejemplo = tf.train.Example(features=tf.train.Features(feature={
    "tokens": tf.train.Feature(int64_list=tf.train.Int64List(value=[5, 9, 2, 7]))
}))
parseado = tf.io.parse_single_example(ejemplo.SerializeToString(), spec_var)
denso = tf.sparse.to_dense(parseado["tokens"])
print("VarLenFeature -> SparseTensor -> denso:", denso.numpy())

## Ejercicios

1. **Escribir**: convertí un subconjunto de Fashion-MNIST a varios shards
   TFRecord; cada `Example` con `image` (bytes) y `label` (int64).
2. **Leer y parsear**: `TFRecordDataset(glob(...)).map(parse_fn)` e iterá
   verificando shapes.
3. **Compresión**: escribí con `TFRecordOptions(compression_type='GZIP')` y
   comparación de tamaño.
4. **Reads paralelos**: usá `interleave` sobre los shards y medí el speedup vs
   lectura serial.
5. **Schema**: comparación de `FixedLenFeature` (tamaño fijo) vs `VarLenFeature`
   (devuelve `SparseTensor`).

## Conclusiones

- `tf.train.Example` envuelve un dict de `Feature` de tres tipos: `BytesList`, `Int64List`, `FloatList`.
- El ciclo es: `SerializeToString` → `TFRecordWriter.write` → `TFRecordDataset` → `parse_single_example`.
- El schema de parseo se declara con `FixedLenFeature` (tamaño fijo) o `VarLenFeature` (variable → `SparseTensor`).
- **Sharding** + `interleave` habilita lecturas paralelas; GZIP/ZLIB reducen el I/O.
- TFRecord brilla cuando los datos no caben en RAM (TPU/Vertex AI); para datasets chicos, NumPy alcanza.